# Session 8 — LLM API Access with the OpenAI SDK

This notebook is OpenAI SDK-first. We will use the same mental model to reference OpenAI, GitHub Models, and Ollama, then briefly compare that to Claude's Python SDK.

## Learning Goals

- use the OpenAI SDK from Python
- understand Chat Completions vs Responses API
- see how GitHub Models and Ollama can fit the same client pattern
- understand that provider-compatible does not mean provider-identical

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import anthropic

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")

print("OpenAI key present:", bool(OPENAI_API_KEY))
print("GitHub token present:", bool(GITHUB_TOKEN))
print("Anthropic key present:", bool(ANTHROPIC_API_KEY))
print("Ollama base URL:", OLLAMA_BASE_URL)

## Helper Functions

The functions below keep the live API calls small and explicit. If you do not have the required key or local service, skip the relevant cells.

In [ ]:
def make_openai_client(api_key=None, base_url=None):
    kwargs = {}
    if api_key:
        kwargs["api_key"] = api_key
    if base_url:
        kwargs["base_url"] = base_url
    return OpenAI(**kwargs)


def require_env(name: str):
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value

## 1. Chat Completions API

Chat Completions uses an explicit `messages` array. It is still common in tutorials and production code.

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = make_openai_client(api_key=require_env("OPENAI_API_KEY"))

chat_response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0,
    messages=[
        {"role": "system", "content": "You are a concise teaching assistant."},
        {"role": "user", "content": "In one sentence, what is retrieval-augmented generation?"},
    ],
)

print(chat_response.choices[0].message.content)

## 2. Responses API

The Responses API separates `instructions` from `input`, which is often cleaner for application code.

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = make_openai_client(api_key=require_env("OPENAI_API_KEY"))

responses_result = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a concise teaching assistant.",
    input="In one sentence, what is retrieval-augmented generation?",
)

print(responses_result.output_text)

### Quick Comparison

- Chat Completions: explicit message history
- Responses API: cleaner top-level structure
- Both are useful; Session 8 emphasizes the Responses API for newer app workflows

## 3. Structured Output

Applications often need machine-readable output, not just free text.

In [ ]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = make_openai_client(api_key=require_env("OPENAI_API_KEY"))

structured = client.responses.create(
    model="gpt-4o-mini",
    input="Policy question: What is RAG and when should we use it?",
    text={
        "format": {
            "type": "json_schema",
            "name": "session8_summary",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string"},
                    "use_case": {"type": "string"}
                },
                "required": ["topic", "use_case"],
                "additionalProperties": False
            }
        }
    },
)

print(structured.output_text)

## 4. Provider Swap Pattern

The OpenAI SDK can also be used with compatible endpoints. This is useful for GitHub Models and Ollama.

In [ ]:
provider_examples = {
    "openai": {
        "api_key_env": "OPENAI_API_KEY",
        "base_url": None,
        "model": "gpt-4o-mini",
    },
    "github_models": {
        "api_key_env": "GITHUB_TOKEN",
        "base_url": "https://models.inference.ai.azure.com",
        "model": "gpt-4o-mini",
    },
    "ollama": {
        "api_key_env": None,
        "base_url": OLLAMA_BASE_URL,
        "model": "llama3.2",
    },
}

provider_examples

In [ ]:
# Change provider_name to "openai", "github_models", or "ollama".
# This demonstrates the compatibility pattern. Live execution still depends on access.

provider_name = "openai"
provider = provider_examples[provider_name]

provider_api_key = os.getenv(provider["api_key_env"]) if provider["api_key_env"] else "ollama"
provider_client = make_openai_client(api_key=provider_api_key, base_url=provider["base_url"])

print(provider_name)
print(provider["base_url"])
print(provider["model"])

# Uncomment to run when the provider is available:
# result = provider_client.responses.create(
#     model=provider["model"],
#     instructions="You are a concise assistant.",
#     input="Give one sentence explaining embeddings.",
# )
# print(result.output_text)

## 5. Minimal Claude Comparison

Claude uses a different Python SDK and message structure. The main point here is contrast, not building a second full workflow.

In [ ]:
# Requires ANTHROPIC_API_KEY
# Skip this cell if you do not have Claude access.

anthropic_client = anthropic.Anthropic(api_key=require_env("ANTHROPIC_API_KEY"))

claude_response = anthropic_client.messages.create(
    model="claude-3-5-sonnet-latest",
    max_tokens=120,
    system="You are a concise teaching assistant.",
    messages=[
        {"role": "user", "content": "In one sentence, what is chunking in RAG?"}
    ],
)

print(claude_response.content[0].text)

## Recap

- We used the OpenAI SDK as the main teaching interface.
- Chat Completions and Responses API solve similar tasks with different shapes.
- GitHub Models and Ollama can often reuse the same client pattern.
- Claude is relevant, but it uses a different SDK and should be treated as a comparison point here.

The next notebook uses these ideas to build a minimal RAG pipeline over PDF documents.